In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from aeon.visualisation import plot_critical_difference
from tscbench.utils import S3FileCache

In [ ]:
#import boto3
#s3 = boto3.client("s3")
#s3.create_bucket(Bucket="tsc-bench")
#s3.put_object(Bucket="tsc-bench", Key="performance-benchmarking/.keep", Body=b"")


In [ ]:
S3_URI = "s3://tsc-bench/performance-benchmarking"

cache = S3FileCache(S3_URI)
print(f"Files on S3: {cache.list_files()}")

results = cache.read_all_parquet()
results

In [ ]:
results['versions'].head(1).item()

In [ ]:
accuracy = (
    results
    .with_columns(
        pl.struct(["y_true", "y_pred"])
        .map_elements(
            lambda x: sum(a == b for a, b in zip(x["y_true"], x["y_pred"])) / len(x["y_true"]),
            return_dtype=pl.Float64,
        )
        .alias("accuracy")
    )
    .group_by(["dataset", "model"])
    .agg(pl.col("accuracy").mean())
    .pivot(values="accuracy", index="dataset", on="model")
    .drop_nulls()
)

clsf = accuracy.select([c for c in accuracy.columns if c != "dataset"])
plot_critical_difference(clsf.to_numpy(), clsf.columns)
plt.tight_layout()
plt.show()